# Aiscern Video Frame Detector — Fine-Tuning
**Platform:** Google Colab (T4 GPU)  
**Target:** `saghi776/aiscern-video-detector`  
**Base model:** `google/vit-base-patch16-224-in21k`  
**Expected accuracy:** >82% on FaceForensics++ frames  
**Runtime:** ~60 minutes

### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Add `HF_TOKEN` in the 🔑 Secrets sidebar
3. Create HF repo: https://huggingface.co/new?owner=saghi776&name=aiscern-video-detector

⚠️ **Colab disconnects kill training.** This notebook saves checkpoints to Google Drive every 200 steps. If it disconnects, re-run from Cell 8 (it will resume automatically).

In [ ]:
# CELL 1 — Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/aiscern-video-checkpoints'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'✅ Checkpoints will save to: {CHECKPOINT_DIR}')

In [ ]:
# CELL 2 — Install
!pip install -q transformers datasets accelerate huggingface_hub pillow scikit-learn

In [ ]:
# CELL 3 — Authenticate
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('✅ Authenticated')

In [ ]:
# CELL 4 — Configuration
BASE_MODEL  = 'google/vit-base-patch16-224-in21k'
REPO_ID     = 'saghi776/aiscern-video-detector'
BATCH_SIZE  = 16    # Colab T4 = 15GB VRAM
EPOCHS      = 4
LR          = 2e-5
IMG_SIZE    = 224
SEED        = 42
MAX_SAMPLES = 15000  # 7.5K real + 7.5K deepfake

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB' if torch.cuda.is_available() else '')

In [ ]:
# CELL 5 — Load deepfake face datasets
# Strategy: use image classification datasets of face crops from deepfake videos
# These are already extracted frames, no video processing needed
from datasets import load_dataset, concatenate_datasets
from PIL import Image
import numpy as np

def normalise_label(val):
    s = str(val).lower().strip()
    if s in ('fake', 'deepfake', 'ai', '1', 'generated', 'synthetic', 'manipulated'): return 1
    if s in ('real', 'authentic', 'genuine', '0', 'human', 'original'): return 0
    return -1

all_splits = []

# Dataset 1: DeepFake-Vs-Real-Faces (face crops)
try:
    ds1 = load_dataset('arnabdhar/DeepFake-Vs-Real-Faces', split='train', token=HF_TOKEN)
    img_col = next((c for c in ds1.column_names if c in ('image','img','face')), None)
    lbl_col = next((c for c in ds1.column_names if 'label' in c.lower()), None)
    if img_col and lbl_col:
        if img_col != 'image': ds1 = ds1.rename_column(img_col, 'image')
        ds1 = ds1.map(lambda x: {'label': normalise_label(x[lbl_col])})
        ds1 = ds1.filter(lambda x: x['label'] != -1)
        ds1 = ds1.select_columns(['image', 'label'])
        all_splits.append(ds1)
        r = sum(1 for x in ds1 if x['label']==0)
        f = len(ds1) - r
        print(f'DeepFake-Vs-Real: {len(ds1):,} (real={r:,} fake={f:,})')
except Exception as e:
    print(f'DeepFake-Vs-Real skipped: {e}')

# Dataset 2: marcelomoreno26/deepfake-detection (FaceForensics++ based)
try:
    ds2 = load_dataset('marcelomoreno26/deepfake-detection', split='train', token=HF_TOKEN)
    img_col = next((c for c in ds2.column_names if c in ('image','img')), None)
    lbl_col = next((c for c in ds2.column_names if 'label' in c.lower()), None)
    if img_col and lbl_col:
        if img_col != 'image': ds2 = ds2.rename_column(img_col, 'image')
        ds2 = ds2.map(lambda x: {'label': normalise_label(x[lbl_col])})
        ds2 = ds2.filter(lambda x: x['label'] != -1)
        ds2 = ds2.select_columns(['image', 'label'])
        all_splits.append(ds2)
        r = sum(1 for x in ds2 if x['label']==0)
        f = len(ds2) - r
        print(f'FaceForensics++: {len(ds2):,} (real={r:,} fake={f:,})')
except Exception as e:
    print(f'FaceForensics++ skipped: {e}')

# Dataset 3: CIFAKE as fallback (AI images vs real photos — close enough)
try:
    ds3 = load_dataset('jlbaker361/cifake-real-and-ai-generated-small-images', split='train')
    ds3 = ds3.select_columns(['image', 'label'])
    all_splits.append(ds3)
    print(f'CIFAKE fallback: {len(ds3):,} samples')
except Exception as e:
    print(f'CIFAKE skipped: {e}')

if not all_splits:
    raise RuntimeError('No datasets loaded — check HF_TOKEN')

combined = concatenate_datasets(all_splits)
print(f'\nTotal: {len(combined):,}')

In [ ]:
# CELL 6 — Balance + split
combined = combined.shuffle(SEED)
real = combined.filter(lambda x: x['label'] == 0)
fake = combined.filter(lambda x: x['label'] == 1)

# Hard assert: labels must be exactly 0 and 1
assert set(combined.unique('label')) <= {0, 1}, f'Unexpected labels: {combined.unique("label")}'

n = min(len(real), len(fake), MAX_SAMPLES // 2)
balanced = concatenate_datasets([
    real.shuffle(SEED).select(range(n)),
    fake.shuffle(SEED).select(range(n)),
]).shuffle(SEED)

split    = balanced.train_test_split(test_size=0.15, seed=SEED)
train_ds = split['train']
val_test = split['test'].train_test_split(test_size=0.5, seed=SEED)
val_ds   = val_test['train']
test_ds  = val_test['test']

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

In [ ]:
# CELL 7 — Preprocess images
from transformers import ViTImageProcessor
from PIL import Image
import random

processor = ViTImageProcessor.from_pretrained(BASE_MODEL)

def augment_and_preprocess(examples, augment=False):
    images = []
    for img in examples['image']:
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img, dtype=np.uint8))
        img = img.convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        if augment:
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
        images.append(img)
    inputs = processor(images=images, return_tensors='pt')
    inputs['labels'] = examples['label']
    return inputs

# Training gets augmentation, val/test do not
print('Preprocessing train...')
train_ds = train_ds.map(lambda x: augment_and_preprocess(x, augment=True),
                         batched=True, batch_size=128, remove_columns=['image'], desc='Train')
print('Preprocessing val...')
val_ds   = val_ds.map(lambda x: augment_and_preprocess(x, augment=False),
                       batched=True, batch_size=128, remove_columns=['image'], desc='Val')
print('Preprocessing test...')
test_ds  = test_ds.map(lambda x: augment_and_preprocess(x, augment=False),
                        batched=True, batch_size=128, remove_columns=['image'], desc='Test')

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('✅ Preprocessing complete')

In [ ]:
# CELL 8 — Load model
from transformers import ViTForImageClassification

# CRITICAL: id2label for hf-analyze.ts compatibility
# /ai|fake|deepfake|generated/i → 'DEEPFAKE'
# /real|authentic|genuine/i      → 'REAL'
model = ViTForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'REAL', 1: 'DEEPFAKE'},
    label2id={'REAL': 0, 'DEEPFAKE': 1},
    ignore_mismatched_sizes=True,
)
print(f'Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

In [ ]:
# CELL 9 — Training (saves to Drive every 200 steps for Colab safety)
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, average='binary')),
    }

training_args = TrainingArguments(
    output_dir              = CHECKPOINT_DIR,   # Drive — survives disconnects
    num_train_epochs        = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate           = LR,
    warmup_ratio            = 0.1,
    weight_decay            = 0.01,
    evaluation_strategy     = 'steps',
    eval_steps              = 200,              # Eval every 200 steps — Colab safety
    save_strategy           = 'steps',
    save_steps              = 200,
    save_total_limit        = 3,
    load_best_model_at_end  = True,
    metric_for_best_model   = 'f1',
    greater_is_better       = True,
    fp16                    = True,
    push_to_hub             = False,            # Push manually in next cell
    report_to               = 'none',
    seed                    = SEED,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

# Auto-resume from checkpoint if Colab disconnected
import os, glob
checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'))
last_checkpoint = checkpoints[-1] if checkpoints else None
if last_checkpoint:
    print(f'Resuming from: {last_checkpoint}')
else:
    print('Starting fresh training...')

trainer.train(resume_from_checkpoint=last_checkpoint)
print('✅ Training complete')

In [ ]:
# CELL 10 — Evaluate
test_results = trainer.evaluate(test_ds)
print(f"\n{'='*40}")
print('TEST RESULTS')
print(f"{'='*40}")
print(f"Accuracy: {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.1f}%)")
print(f"F1:       {test_results['eval_f1']:.4f}")
print(f"\nTarget (>82%): {'✅' if test_results['eval_accuracy'] >= 0.82 else '⚠️ '}")

In [ ]:
# CELL 11 — Push to HuggingFace
# Run this cell even if Colab disconnected — it just pushes the best checkpoint
from transformers import ViTForImageClassification, ViTImageProcessor
from huggingface_hub import login

# Re-auth in case session restarted
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

acc = test_results['eval_accuracy']
f1  = test_results['eval_f1']

trainer.model.push_to_hub(
    REPO_ID,
    commit_message=f'ViT deepfake frame classifier — acc={acc:.3f} f1={f1:.3f} | Aiscern video detector',
    token=HF_TOKEN,
)
processor.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f'✅ Pushed to https://huggingface.co/{REPO_ID}')